# D0 · Desktop GIS engines — QGIS, GDAL/OGR, GRASS, SAGA from Jupyter

**What group D is:** four big GIS engines that you usually meet as buttons in **QGIS's Processing Toolbox**.
Here you call them from Jupyter, so every step is written down and can be repeated.

| Engine | Best at | Daily picture | Notebook |
|---|---|---|---|
| **QGIS native** | vector work: buffers, overlays, geometry repair, spatial joins, networks | the Swiss-army knife | D1 |
| **GDAL / OGR** | converting, reprojecting and clipping rasters (GDAL) and vectors (OGR) | the universal plug adapter | D2 |
| **GRASS GIS** | hydrology, terrain, raster and environmental modelling | a big scientific laboratory | D3 |
| **SAGA GIS** | terrain and hydrological analysis (wetness, channels, landforms) | a specialist workshop for landscapes | D4 |

**How they relate to what you know:**
- GDAL/OGR is the engine inside **Rasterio, Pyogrio, Fiona and PostGIS rasters**. D2 shows the same engine as command-line tools.
- QGIS native tools use **GEOS** (like Shapely/PostGIS) and **PROJ** (like PyProj).
- GRASS and SAGA add hydrology and terrain methods that GeoPandas and Rasterio do not have.

**Three ways to run a Processing tool:**
1. **QGIS window:** Processing Toolbox → double-click a tool → fill the form. Good for learning; hard to repeat.
2. **`qgis_process` command line:** `qgis_process run native:buffer -- INPUT=... DISTANCE=600 OUTPUT=...`.
   Works from **any** Jupyter, even if its Python is not QGIS's Python. **We use this**, through the helper `qrun()`.
3. **PyQGIS** inside QGIS's own Python: `processing.run("native:buffer", {...})`. Same parameters as option 2.

**Recommendation:** learn the tool in the QGIS window once (see the form), then write it with `qrun()`.
Tip: in any QGIS tool dialog, *Advanced → Copy as qgis_process Command* gives you the exact command.

## Step 1 · Install (choose your system)

| System | QGIS (includes GDAL, GRASS provider) | SAGA | GDAL command line |
|---|---|---|---|
| **Windows** | OSGeo4W installer → "Express install" → QGIS | OSGeo4W advanced install → `saga`; or saga-gis.sourceforge.io | included in OSGeo4W (use the *OSGeo4W Shell*) |
| **macOS** | qgis.org installer (QGIS.app) | `brew install saga-gis` or conda | `brew install gdal` |
| **Linux (Ubuntu)** | `sudo apt install qgis qgis-provider-grass grass` | `sudo apt install saga` | `sudo apt install gdal-bin` |
| **conda (any)** | `conda install -c conda-forge qgis grass` | `conda install -c conda-forge saga` | `conda install -c conda-forge gdal` |

Since QGIS 3.30, SAGA is **not** built into QGIS any more; inside QGIS you can add it with the *Processing Saga NextGen Provider* plugin.
In this course we call SAGA directly with its own command-line tool, `saga_cmd` (D4).

If `qgis_process` is not found, set `QGIS_PROCESS` to its full path before starting Jupyter
(Windows: `C:\OSGeo4W\bin\qgis_process-qgis.bat`; macOS: `/Applications/QGIS.app/Contents/MacOS/bin/qgis_process`).

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

In [ ]:
from geotrain.desktop import qrun, qhelp, qlist, sh, grass_alg, enable_grass, qgis_process_path, quote, minimal_project
OUT = DATA_DIR / "d_out"
OUT.mkdir(exist_ok=True)
V = lambda layer: f"{GPKG}|layername={layer}"       # how QGIS names a layer inside a GeoPackage
DEM = DATA_DIR / "dem.tif"
print("qgis_process:", qgis_process_path())

---
## Level 1 — Check the four engines

**Goal of this level:** confirm each engine answers, and see which versions you have.

**Daily-life picture:** Before a long drive: check oil, water, tyres and fuel.

### Exercise 1.1 — Is QGIS Processing there?

**Step 1 · Purpose.** What is the **purpose** of `qgis_process --version`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Run the shell command with the helper: `sh(f"{quote(qgis_process_path())} --version")`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh(f"{quote(qgis_process_path())} ____")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Prints the versions of QGIS and the libraries inside it (GDAL, GEOS, PROJ…).

**Real life:** A colleague's result differs from yours: first compare versions.

```python
sh(f"{quote(qgis_process_path())} --version")
```

</details>

### Exercise 1.2 — Switch on GRASS inside QGIS

**Step 1 · Purpose.** What is the **purpose** of `qgis_process plugins enable grassprovider`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Call `enable_grass()`, then count GRASS tools with `qlist("r.watershed")`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
enable_grass()
print(qlist("____"))
print("GRASS tool id:", grass_alg("r.watershed"))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Activates the GRASS provider, so GRASS tools appear in QGIS Processing (and in `qgis_process list`).

**Real life:** Needed once on a new computer before using `grass:r.watershed` and friends.

```python
enable_grass()
print(qlist("r.watershed"))
print("GRASS tool id:", grass_alg("r.watershed"))
```

QGIS 3.34 and older call them `grass7:...`, newer versions `grass:...`. `grass_alg()` finds the right name for you.

</details>

### Exercise 1.3 — How many tools per provider?

**Step 1 · Purpose.** What is the **purpose** of `qlist()`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`ids = [r[0] for r in qlist()]`, then count the part before `:` with `collections.Counter`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
from collections import Counter
ids = [r[0] for r in qlist()]
Counter(i.split("____")[0] for i in ids)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Lists all Processing algorithms; each id starts with its provider (`native:`, `gdal:`, `grass7:`/`grass:`, `qgis:`).

**Real life:** Knowing what your toolbox holds before planning an analysis.

```python
from collections import Counter
ids = [r[0] for r in qlist()]
Counter(i.split(":")[0] for i in ids)
```

</details>

### Exercise 1.4 — GDAL and SAGA command-line tools

**Step 1 · Purpose.** What is the **purpose** of `gdalinfo --version / saga_cmd --version`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Two `sh()` calls. SAGA prints a banner; look for 'SAGA Version'.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
sh("gdalinfo ____")
sh("saga_cmd --version")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Checks that the stand-alone GDAL and SAGA programs are on your PATH.

**Real life:** D2 (GDAL) and D4 (SAGA) call these programs directly.

```python
sh("gdalinfo --version")
sh("saga_cmd --version")
```

</details>

### ✅ Setup check

If all four cells ran, go on with **D1** (QGIS native). Every D notebook starts with the same two setup cells.